# PyTorch - Transformers Experimentation

This notebook is intended to experiment the usage of Transformers in PyTorch for Time Series Forecasting.

# Notebook Setup

## Imports

In [20]:
# Import Standard Libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

## Define Configurations

In [21]:
# Data path
sunspot_data_path = './../../data/raw/sunspot_data.csv'

# Read Data

In [22]:
# Read data from local path
sunspot_data = pd.read_csv(
    sunspot_data_path,
    sep=';',
    header=None,
    names=['year', 'month', 'day', 'dec_year', 'sn_value', 'sn_error', 'obs_num', 'unused1'],
    na_values=['-1'],
    index_col=False
)

# Data Preprocessing

## Sunspot Data

In [23]:
# Find the first id that has a subsequent sequence with at least 1 observation
start_id = max(sunspot_data[sunspot_data['obs_num'] == 0].index.tolist()) + 1

# Split train and test data
sunspot_data_valida = sunspot_data.iloc[start_id:].copy()
sunspot_data['sn_value'] = sunspot_data['sn_value'].astype(float)
sunspot_data_train = sunspot_data[sunspot_data['year'] < 2000]
sunspot_data_test = sunspot_data[sunspot_data['year'] >= 2000]

# Select only the column 'sn_value'
sunspot_data_train = sunspot_data_train['sn_value'].to_numpy().reshape(-1, 1)
sunspot_data_test = sunspot_data_test['sn_value'].to_numpy().reshape(-1, 1)

# Standardisation
scaler = StandardScaler()
sunspot_data_train = scaler.fit_transform(sunspot_data_train).flatten().tolist()
sunspot_data_test = scaler.transform(sunspot_data_test).flatten().tolist()

# Create different sequence batches of 10 time steps each
def to_sequences(sequence_size, observations):
    """Transform a sequence of observations into train and test sequences. (e.g., [1, 2, 3] -> [4])"""
    x, y = [], []
    for i in range(len(observations) - sequence_size):
        # Compute the current window and the subsequent element (i.e., target)
        window = observations[i:(i + sequence_size)]
        after_window = observations[i + sequence_size]

        # Append them
        x.append(window)
        y.append(after_window)

    return (torch.tensor(x, dtype=torch.float32).view(-1, sequence_size, 1),
            torch.tensor(y, dtype=torch.float32).view(-1, 1))

# to_sequence example
example_sequence = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
example_sequence_size = 4
example_sequence_output = to_sequences(example_sequence_size, example_sequence)
print('Example sequence:', example_sequence)
print('Example sequence size:', example_sequence_size)
print('Example sequence output X one element:', example_sequence_output[0][0])
print('Example sequence Output Y one element:', example_sequence_output[1][0])

# Transform train and test into sequences
sunspot_sequence_size = 10
x_train, y_train = to_sequences(sunspot_sequence_size, sunspot_data_train)
x_test, y_test = to_sequences(sunspot_sequence_size, sunspot_data_test)

# Create the Data Loader in PyTorch
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Example sequence: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Example sequence size: 4
Example sequence output X one element: tensor([[1.],
        [2.],
        [3.],
        [4.]])
Example sequence Output Y one element: tensor([5.])


# Model Definition

## Positional Encoding with Sinusoidal Functions

In [24]:
class PositionalEncoder(nn.Module):
    """
    Define a Positional Encoder through sinusoidal functions.

    PE(pos, 2i) = sin(pos/(10000^(2i/d_model)))
    PE(pos, 2i + 1) = cost(pos/(10000^(2i/d_model)))

    pos: position in the sequence
    i: is the dimension index (half of the model dimension d_model)
    d_model: is the model dimension (the embedding dimension)

    NOTE: 2i and 2i + 1 is for separate sine and cosine values into even and odd indicies.
    """
    def __init__(self, embeddings_size, dropout_probability=0.1, max_len_sequence=5000):

        # Initialise the super class
        super(PositionalEncoder, self).__init__()

        # Set the dropout layer
        self.dropout = nn.Dropout(p=dropout_probability)

        # Initialise positional encoding matrix of dimension (max_length_sequence x embeddings_size)
        positional_encoding_matrix = torch.zeros(max_len_sequence, embeddings_size)

        # Create the position from 1 to the max length of the input sequence (reshape x -> (x, 1))
        position = torch.arange(0, max_len_sequence, dtype=torch.float).unsqueeze(1)

        # Create the dividend term as 10000^(2i/d)
        dividend_term = torch.exp(torch.arange(0, embeddings_size, 2).float() * (-np.log(10000.0) / embeddings_size))

        # Compute positional encoding for even and odd columns in the Positional Encoding Matrix
        positional_encoding_matrix[:, 0::2] = torch.sin(position * dividend_term)
        positional_encoding_matrix[:, 1::2] = torch.cos(position * dividend_term)

        # Add a dimension for the batch_size through 'unsqueeze(0)' in the first index and then transpose
        # NOTE: (max_length_sequence, embeddings_size) -> (max_length_sequence, batch_size, embeddings_size)
        positional_encoding_matrix = positional_encoding_matrix.unsqueeze(0).transpose(0, 1)

        # Save the PE matrix
        self.register_buffer('positional_encoding_matrix', positional_encoding_matrix)

    def forward(self, sequence):

        # Add the positional encoding to the input sequence (Just sum them up)
        output = sequence + self.positional_encoding_matrix[:sequence.size(0), :]

        # Apply dropout
        return self.dropout(output)


In [25]:
# Example of positional encoding
example_sequence_len = 5
example_embeddings_size = 4

# Initialise variables
example_pe = torch.zeros(example_sequence_len, example_embeddings_size)
example_position = torch.arange(0, example_sequence_len, dtype=torch.float).unsqueeze(1)
example_dividend_term = torch.arange(0, example_embeddings_size, 2).float()

# Compute positional encoding for even and odd columns in the Positional Encoding Matrix
example_pe[:, 0::2] = example_position * example_dividend_term
example_pe[:, 1::2] = (example_position * example_dividend_term) - 1

# Define a sequence
example_sequence = torch.tensor([
    [1, 11, 111, 1111],
    [2, 22, 222, 2222],
    [3, 33, 333, 3333],
    [4, 44, 444, 4444],
    [5, 55, 555, 5555]]
)

print('Sequence Length: ', example_sequence_len)
print('Embeddings Size: ', example_embeddings_size)
print('Position: ', example_position)
print('Dividend term: ', example_dividend_term)
print('Positional Encoding (Pre Transformation): ', example_pe)
print('Positional Encoding (Transformed): ', example_pe.unsqueeze(0).transpose(0, 1))
print('Positional Encoding (Pre Shape)', example_pe.shape)
print('Positional Encoding (After Shape)', example_pe.unsqueeze(0).transpose(0, 1).shape)
print('Sequence: ', example_sequence)
print('Sequence shape: ', example_sequence.shape)

Sequence Length:  5
Embeddings Size:  4
Position:  tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])
Dividend term:  tensor([0., 2.])
Positional Encoding (Pre Transformation):  tensor([[ 0., -1.,  0., -1.],
        [ 0., -1.,  2.,  1.],
        [ 0., -1.,  4.,  3.],
        [ 0., -1.,  6.,  5.],
        [ 0., -1.,  8.,  7.]])
Positional Encoding (Transformed):  tensor([[[ 0., -1.,  0., -1.]],

        [[ 0., -1.,  2.,  1.]],

        [[ 0., -1.,  4.,  3.]],

        [[ 0., -1.,  6.,  5.]],

        [[ 0., -1.,  8.,  7.]]])
Positional Encoding (Pre Shape) torch.Size([5, 4])
Positional Encoding (After Shape) torch.Size([5, 1, 4])
Sequence:  tensor([[   1,   11,  111, 1111],
        [   2,   22,  222, 2222],
        [   3,   33,  333, 3333],
        [   4,   44,  444, 4444],
        [   5,   55,  555, 5555]])
Sequence shape:  torch.Size([5, 4])


## Multi-Head Attention Transformer

![Multi-Head Attention Transformer](./../../docs/images/multi_head_attention_transformer.png)

In [26]:
class TransformerModel(nn.Module):
    """
    Encoder-Decoder transformer model.
    """
    def __init__(self, input_length=1,
                 embeddings_size=64,
                 n_head=4,
                 num_layers=2,
                 dropout_probability=0.2):

        # Initialise the super class
        super(TransformerModel, self).__init__()

        # Define an encoder to transform the input sequence in the embedding size
        self.encoder = nn.Linear(input_length, embeddings_size)

        # Define the positional encoding
        self.pos_encoder = PositionalEncoder(embeddings_size, dropout_probability)

        # It comprises an attention block for encoding the sequence through sel-attention and giving a new "better" representation
        encoder_layers = nn.TransformerEncoderLayer(embeddings_size, n_head)

        # Create multiple sequential encoder layer blocks
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)

        # Final decoder layer to return a single predicted time step value
        self.decoder = nn.Linear(embeddings_size, 1)

    def forward(self, sequence):

        output = self.encoder(sequence)
        output = self.pos_encoder(output)
        output = self.transformer(output)
        output = self.decoder(output[:, -1, :]) # Return only the last time step (a.k.a., the one to predict in the sequence)
        return output


# Model Training

In [27]:
# Instance model
multi_head_attention_transformer = TransformerModel().to(device)

/Users/simone.porreca/Projects/TimeWarpForecast/.venv/lib/python3.13/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


NameError: name 'device' is not defined